# Resume: Validation N=50 do Steering +10pp

Máquina desligou. Artefatos salvos em `caiovicentino1/qwen36-27b-sae-multilayer`:
- 3 SAEs treinadas (L11, L31, L55)
- Feature stats + characterization

**Não re-treina nada**. Só resume com validation escalada pra N=50.

## Config do winner (do run anterior, 20 prompts)

```python
STEER_FEATURES = {11: 4053, 31: 124, 55: 590}
STEER_ALPHA = 1.0  # all correct↑ amplify
```

Resultado n=20: **combined +10pp** (0.75 → 0.85).

## Now: N=50 replication + wrong↓ supression test

Roda mesmo protocolo em 50 prompts, tight CI. Também testa wrong↓ suppression variant pra ver se adiciona.

**Budget**: ~5-6h (50 prompts × 3-4 conditions).

## Cell 1 — Setup (skip training)

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5' in CONFIG_MAPPING_NAMES
except Exception: ok = False
if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from collections import defaultdict, Counter
from safetensors import safe_open
from huggingface_hub import snapshot_download, HfApi
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/steering_val'); OUT.mkdir(exist_ok=True)
MODEL_ID = 'Qwen/Qwen3.6-27B'
HF_SAE = 'caiovicentino1/qwen36-27b-sae-multilayer'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'

LAYERS = [11, 31, 55]
D_MODEL = 5120
N_FEATURES = 4096
TOPK = 32

# Winning config from n=20 run
STEER_CORRECT = {11: 4053, 31: 124, 55: 590}  # correct↑ amplify, α=+1.0

# Also test wrong↓ suppression
STEER_WRONG = {
    11: [(2503, -1.5)],            # "perfectly/plausible" overconfidence
    31: [(452, -0.5)],              # domain confusion
    55: [],                          # skip (too noisy)
}

ALPHA_CORRECT = 1.0
MAX_NEW = 2500
N_EVAL = 50
FORCE_SUFFIX = '\n</think>\n\nFinal answer: \\boxed{'

print('setup done: N=50 validation of +10pp steering')

## Cell 2 — Load model + 3 SAEs from HF

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'27B loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Download SAEs
sae_dir = snapshot_download(HF_SAE, repo_type='model', cache_dir='/content/cache',
                             allow_patterns=['sae_L*.pt'])

class TopKSAE(nn.Module):
    def __init__(self, d_in, n, k):
        super().__init__()
        self.W_enc = nn.Parameter(torch.zeros(d_in, n))
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.W_dec = nn.Parameter(torch.zeros(n, d_in))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        top_v, top_i = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre); z.scatter_(-1, top_i, F.relu(top_v))
        return z, top_i, top_v

saes = {}
for L in LAYERS:
    sae = TopKSAE(D_MODEL, N_FEATURES, TOPK)
    sd = torch.load(f'{sae_dir}/sae_L{L}_n{N_FEATURES}_k{TOPK}.pt',
                    map_location='cpu', weights_only=True)
    sae.load_state_dict(sd)
    for p in sae.parameters(): p.requires_grad_(False)
    saes[L] = sae.cuda()
    print(f'SAE L{L} loaded')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

layer_modules = {L: get_layer_module(model, L) for L in LAYERS}

## Cell 3 — Install steering hooks (flexible: correct↑ / wrong↓ / both)

In [ ]:
steer_config = {'mode': 'off'}  # 'off' / 'correct' / 'wrong' / 'both'

# Pre-compute W_dec rows for steering
W_dec_correct = {L: saes[L].W_dec[STEER_CORRECT[L]].clone().cuda() for L in LAYERS}
W_dec_wrong = {}
for L in LAYERS:
    vecs = []
    for fi, alpha in STEER_WRONG.get(L, []):
        w = saes[L].W_dec[fi].clone().cuda() * alpha
        vecs.append(w)
    W_dec_wrong[L] = sum(vecs) if vecs else torch.zeros(D_MODEL, device='cuda')

def make_steer_hook(L):
    def h(mod, inp, out):
        mode = steer_config['mode']
        if mode == 'off': return out
        x = out[0] if isinstance(out, tuple) else out
        delta = torch.zeros(D_MODEL, device=x.device, dtype=x.dtype)
        if mode in ('correct', 'both'):
            delta = delta + (ALPHA_CORRECT * W_dec_correct[L]).to(x.dtype)
        if mode in ('wrong', 'both'):
            delta = delta + W_dec_wrong[L].to(x.dtype)
        x_new = x.clone()
        x_new[:, -1, :] += delta
        if isinstance(out, tuple): return (x_new,) + out[1:]
        return x_new
    return h

handles = []
for L in LAYERS:
    h = layer_modules[L].register_forward_hook(make_steer_hook(L))
    handles.append(h)
print(f'OK {len(handles)} steering hooks installed')

## Cell 4 — Helpers + load eval prompts

In [ ]:
def extract_answer(text):
    post = text.split('</think>')[-1] if '</think>' in text else text
    for pattern in [r'\\boxed\{([A-J])\}',
                    r'[Aa]nswer\s*(?:is\s*)?[:=]?\s*\*?\*?\(?([A-J])\)?\*?\*?']:
        m = re.search(pattern, post, re.MULTILINE)
        if m: return m.group(1).upper()
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    if m: return m[-1]
    return None

def fmt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = (f'Answer the following multiple-choice question. '
               f'Reason step by step, then put the letter in \\boxed{{}}.\n\n'
               f'Question: {q}\n\nOptions:\n{choices}')
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=True)

def force_close(full_ids, prompt_len):
    gen = tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=False)
    if '</think>' in gen:
        return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True)
    full = tok.decode(full_ids.tolist(), skip_special_tokens=False) + FORCE_SUFFIX
    ctx = tok(full, return_tensors='pt', add_special_tokens=False).input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ctx, max_new_tokens=15, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    suf = tok.decode(out[0, ctx.shape[1]:].tolist(), skip_special_tokens=True)
    return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True) + FORCE_SUFFIX + suf

def gen_answer(q, mode):
    steer_config['mode'] = mode
    p = fmt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    steer_config['mode'] = 'off'
    return extract_answer(force_close(out[0], ids.shape[1]))

# Load eval prompts — use SAME seed 777 as original run but more prompts
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))
questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

# Reserve seed 100 (SAE training) + seed 42 (probe training) first positions
random.seed(100)
questions_100 = questions.copy()
random.shuffle(questions_100)
used_100 = set(q['question'] for q in questions_100[:150])

random.seed(42)
questions_42 = questions.copy()
random.shuffle(questions_42)
used_42 = set(q['question'] for q in questions_42[:100])

# Eval with seed 777, exclude used
random.seed(777)
fresh = [q for q in questions if q['question'] not in used_100 and q['question'] not in used_42]
random.shuffle(fresh)
eval_set = fresh[:N_EVAL]
print(f'{len(eval_set)} eval prompts (seed 777, disjoint from training seeds)')

## Cell 5 — Run 4 conditions × 50 prompts (crash-safe)

In [ ]:
results = {'baseline': [], 'correct_amplify': [], 'wrong_suppress': [], 'both': []}
t0 = time.time()

for qi, q in enumerate(tqdm(eval_set, desc='N=50 steering val')):
    gold = q['gold_letter']
    for cond_name, mode in [('baseline', 'off'), ('correct_amplify', 'correct'),
                              ('wrong_suppress', 'wrong'), ('both', 'both')]:
        try:
            ans = gen_answer(q, mode)
            results[cond_name].append((ans, ans == gold))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); results[cond_name].append((None, False))
        except Exception as e:
            results[cond_name].append((None, False))

    if (qi+1) % 3 == 0:
        print(f'  [{qi+1}/{len(eval_set)}]', end=' ')
        for k in results:
            acc = np.mean([r[1] for r in results[k]])
            print(f'{k[:8]}={acc:.3f}', end=' ')
        print()
        # Crash-safe partial
        partial = {k: [(r[0], bool(r[1])) for r in v] for k, v in results.items()}
        with open(OUT / 'partial.json', 'w') as f:
            json.dump(partial, f, indent=2)

for h in handles: h.remove()

b = np.mean([r[1] for r in results['baseline']])
print(f'\n=== FINAL (n={len(eval_set)}, {(time.time()-t0)/60:.0f} min) ===')
for k in results:
    acc = np.mean([r[1] for r in results[k]])
    delta = (acc - b) * 100 if k != 'baseline' else 0
    print(f'  {k:18s}: {acc:.3f}  Δ={delta:+.1f}pp')

## Cell 6 — Upload validation

In [ ]:
summary = {
    'model': MODEL_ID,
    'experiment': 'N50_steering_validation',
    'n_eval': len(eval_set),
    'eval_seed': 777,
    'steer_config': {'correct': STEER_CORRECT, 'wrong': {str(L): v for L, v in STEER_WRONG.items()}},
    'results': {k: float(np.mean([r[1] for r in v])) for k, v in results.items()},
    'deltas_pp': {k: float((np.mean([r[1] for r in v]) - b) * 100) for k, v in results.items() if k != 'baseline'},
    'original_n20': {'baseline': 0.75, 'combined': 0.85, 'delta_pp': 10.0},
    'eval_date': time.strftime('%Y-%m-%d'),
}
with open(OUT / 'validation_n50.json', 'w') as f:
    json.dump(summary, f, indent=2)

api = HfApi()
api.upload_file(path_or_fileobj=str(OUT/'validation_n50.json'),
                path_in_repo='validation_n50.json',
                repo_id=HF_SAE, repo_type='model',
                commit_message='N=50 steering validation')
api.upload_file(path_or_fileobj=str(OUT/'partial.json'),
                path_in_repo='validation_n50_partial.json',
                repo_id=HF_SAE, repo_type='model',
                commit_message='Per-prompt validation details')
print(f'\n✅ https://huggingface.co/{HF_SAE}')